# Hydranomic - Modelo Preditivo de Consumo de Água (Machine Learning)

Notebook especializado para experimentação, treinamento e avaliação do modelo preditivo de consumo de água (`RandomForestRegressor`).
Compatível com o ambiente de Notebooks da **WEGnology**.

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Importa funções do módulo de ML adaptado
from machinelearning import (
    carregar_dataframe,
    preparar_dados,
    treinar_modelo,
    prever_proximo_ciclo,
    executar_pipeline_ml
)
from wegnology_adapter import salvar_json

## 1. Carregamento e Inspeção dos Dados Históricos

In [ ]:
df_bruto = carregar_dataframe()
print(f"Formato dos dados: {df_bruto.shape}")
df_bruto.info()
df_bruto.head()

## 2. Engenharia de Variáveis (Features)

In [ ]:
df_completo, df_treino, features_x, alvo_y = preparar_dados(df_bruto)
print("Features de Entrada:", features_x)
print("Variável Alvo:", alvo_y)
df_treino[['id', 'unidade_consumidora', 'mes', 'tempo_iniciofinal', 'memoria_consumo', 'consumo_m3']].head()

## 3. Treinamento e Avaliação do RandomForestRegressor

In [ ]:
modelo = treinar_modelo(df_treino, features_x, alvo_y)

# Previsões na base de treino para diagnóstico
predicoes_treino = modelo.predict(df_treino[features_x])
mae = mean_absolute_error(df_treino[alvo_y], predicoes_treino)
rmse = np.sqrt(mean_squared_error(df_treino[alvo_y], predicoes_treino))
r2 = r2_score(df_treino[alvo_y], predicoes_treino)

print(f"MAE (Erro Médio Absoluto): {mae:.2f} m³")
print(f"RMSE (Raiz do Erro Quadrático Médio): {rmse:.2f} m³")
print(f"R² Score: {r2:.4f}")

## 4. Gráfico de Consumo Real vs Previsão do Modelo

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(df_treino['id'], df_treino[alvo_y], marker='o', label='Consumo Real (m³)', color='#0ea5e9')
plt.plot(df_treino['id'], predicoes_treino, marker='x', linestyle='--', label='Predição do Modelo (m³)', color='#f97316')
plt.title("Comparativo Histórico: Consumo Real vs Ajuste do Modelo de ML")
plt.xlabel("Período (Ano-Mês)")
plt.ylabel("Consumo (m³)")
plt.xticks(rotation=45)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Projeção para o Próximo Ciclo e Salvamento do JSON

In [ ]:
resultado = prever_proximo_ciclo(modelo, df_completo, features_x, tarifa=59.6)
caminho_salvo = salvar_json("previsao_ml.json", resultado)

print("Resultado salvo no OUTPUT_DIR para a WEGnology:")
print(resultado)